In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
import scipy.stats as stats
import statsmodels.api as sm 

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor 
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [ ]:
df = pd.read_pickle("instagram_node_dataframe.pkl")
df.head()

### Datenaufbereitung für die Analyse
Hier werden für die Analyse relevante Daten aufbereitet und passendere Spaltennamen vergeben. 

Im df_gefiltert Dataframe enthält die wichtigsten Spalten, die einen Einfluss auf Likeanzahl haben könnten. 


In [ ]:
df_gefiltert = df.rename(columns={
    "edge_media_preview_like.count": "likes",
    "owner.edge_followed_by.count": "follower_count",
    "owner.is_verified": "is_verified",
    "owner.category_name": "account_category",
    "owner.is_business_account": "is_business_account",
    "owner.is_professional_account": "is_professional_account",
    "edge_media_to_comment.count": "comment_count",
    "edge_media_to_tagged_user.edges": "tagged_users",
    "edge_media_to_caption.edges": "caption_text",
    "__typename": "post_type",
    "edge_sidecar_to_children.edges": "is_carousel",
    "taken_at_timestamp": "timestamp"
    })

df_gefiltert = df_gefiltert[[
    "likes",
    "follower_count",
    "is_verified",
    "account_category",
    "is_business_account",
    "is_professional_account",
    "comment_count",
    "tagged_users",
    "caption_text",
    "post_type",
    "is_carousel",
    "timestamp"
]]
df_gefiltert.head()

In [ ]:
df_gefiltert.info()

### Vorbereitung

#### Zielvariable: Likes

Die Rohdaten der "Likes"-Anzahl weisen oft Eigenschaften auf (z.B. extreme Schiefe, Ausreißer), die statistische Analysen oder die Leistung von Machine-Learning-Modellen beeinträchtigen können. Ziel der folgenden Schritte ist es, die Datenqualität zu verbessern und die Variable für nachfolgende Analysen zu optimieren.

- Negative Werte behandeln
    - Alle Werte kleiner als 0 werden auf 0 gesetzt
    - "Likes" sind Zählwerte und können logischerweise nicht negativ sein. Dieser Schritt korrigiert mögliche fehlerhafte negative Einträge.

-  Log-Transformation
    - Die Log-Transformation dient dazu, die statistischen Eigenschaften der Gesamtheit der Daten zu verbessern und sie für die Modellierung oder bestimmte Analysen geeigneter zu machen. Sie zielt darauf ab, Muster und Beziehungen aufzudecken, die in den Rohdaten durch die Schiefe oder Ausreißer verdeckt sein könnten.



In [ ]:
df_gefiltert['likes'] = df_gefiltert['likes'].clip(lower=0).astype(int)
# Log-Transformation der Likes
df_gefiltert['likes_log'] = np.log1p(df_gefiltert['likes'])
print("\n'likes_log' Spalte erstellt.")
df_gefiltert[['likes_log', 'likes']]

Zeitbasierte Merkmale

In [ ]:
df_gefiltert['datetime'] = pd.to_datetime(df_gefiltert['timestamp'], unit='s')
df_gefiltert['hour_of_day'] = df_gefiltert['datetime'].dt.hour
df_gefiltert['day_of_week'] = df_gefiltert['datetime'].dt.day_name(locale='de_DE')  
df_gefiltert['month'] = df_gefiltert['datetime'].dt.month_name(locale='de_DE')
      
print("\nZeitbasierte Merkmale erstellt.")
df_gefiltert[['datetime', 'hour_of_day', 'day_of_week', 'month']].head()

caption_text Extraktion


In [ ]:
def extract_caption(edges):
    if isinstance(edges, list) and len(edges) > 0 and 'node' in edges[0] and 'text' in edges[0]['node']:
        return edges[0]['node']['text']
    return "" 

df_gefiltert['actual_caption_text'] = df_gefiltert['caption_text'].apply(extract_caption)
df_gefiltert['caption_length'] = df_gefiltert['actual_caption_text'].apply(len)
df_gefiltert['hashtag_count'] = df_gefiltert['actual_caption_text'].str.count('#')
df_gefiltert['mention_count'] = df_gefiltert['actual_caption_text'].str.count('@')

df_gefiltert[['actual_caption_text', 'caption_length', 'hashtag_count', 'mention_count']].head()

Karussell-Informationen

In [ ]:
def check_carousel_and_count(edges):
    if isinstance(edges, list) and len(edges) > 0:
        return True, len(edges) # Ist Karussell, Anzahl der Medien
    return False, 1 # Kein Karussell (oder einzelnes Medium)

results = df_gefiltert['is_carousel'].apply(lambda x: check_carousel_and_count(x)) 
df_gefiltert['is_carousel_bool'] = [res[0] for res in results]
df_gefiltert['media_count_in_post'] = [res[1] for res in results]

# Für Nicht-Karussell-Posts ('GraphImage', (kein 'GraphSidecar/Karusell')) ist media_count_in_post typischerweise 1
df_gefiltert.loc[df_gefiltert['post_type'] != 'GraphSidecar', 'media_count_in_post'] = 1
df_gefiltert.loc[df_gefiltert['post_type'] != 'GraphSidecar', 'is_carousel_bool'] = False

df_gefiltert[['is_carousel_bool', 'media_count_in_post']].head()

### Bereinigung des DataFrames für die Modellierung

In [ ]:
# Auswahl der Features für das Modell
feature_columns = [
    'follower_count', 'is_verified', 'account_category', 'is_business_account',
    'is_professional_account', 'comment_count', 'post_type',
    'caption_length', 'hashtag_count', 'mention_count', 'tagged_user_count', 
    'media_count_in_post', 'has_location',
    'hour_of_day', 'day_of_week', 'month'
]
existing_feature_columns = [col for col in feature_columns if col in df_gefiltert.columns]
      

X = df_gefiltert[existing_feature_columns].copy()
y = df_gefiltert['likes_log'].copy()

print("\nAusgewählte Features für X:")
X.info()

# Fehlende Werte in 'account_category' mit 'Unbekannt' füllen 
if 'account_category' in X.columns:
    X['account_category'] = X['account_category'].fillna('Unbekannt')
    
print("\nFehlende Werte in X nach Imputation für 'account_category':")
print(X.isnull().sum())

X

# Explorative Datenanalyse (EDA) der aufbereiteten Metadaten

### Verteilung der Zielvariablen (likes_log)

Histogramme nebeneinander zum Vergleich der Verteilungen von "likes_log" und "likes"



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(df_gefiltert['likes_log'], kde=True, bins=50, ax=axes[0])
axes[0].set_title('Verteilung der log-transformierten Likes (log(1+Likes))')
axes[0].set_xlabel('Log(1 + Anzahl Likes)')
axes[0].set_ylabel('Häufigkeit')

sns.histplot(df_gefiltert['likes'], kde=True, bins=50, ax=axes[1])
axes[1].set_title('Verteilung der Likes')
axes[1].set_xlabel('Anzahl Likes')
axes[1].set_ylabel('Häufigkeit')

plt.tight_layout()
plt.show()



Boxplots nebeneinander zum Vergleich


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(x=df_gefiltert['likes_log'], ax=axes[0])
axes[0].set_title('Boxplot der log-transformierten Likes')
axes[0].set_xlabel('Log(1 + Anzahl Likes)')

sns.boxplot(x=df_gefiltert['likes'], ax=axes[1])
axes[1].set_title('Boxplot der Likes')
axes[1].set_xlabel('Anzahl Likes')

plt.tight_layout()
plt.show()

### Korrelationen numerischer Merkmale

In [ ]:
num_merkmale_corr = [col for col in X.select_dtypes(include=np.number).columns if col in X.columns]

temp_df_corr = X[num_merkmale_corr].copy()
temp_df_corr['likes_log'] = y 

correlation_matrix = temp_df_corr.corr()
plt.figure(figsize=(16, 12))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f",  cmap="RdBu", center = 0)
plt.title('Korrelationsmatrix der numerischen Features und log(Likes)')
plt.show()

### Interpretation von Korrelationsmatrix:

Um eine Überblick über die einzelne prädikatoren zu erhalten, wird ein pairplot erstellt, der die Korrelationen zwischen den numerischen Merkmalen visualisiert. 
Die Likes-Log-Transformation wird als Zielvariable in Abhängigkeit von `follower_count`, `comment_count`, 
`caption_length`, `hashtag_count`, `mention_count`, `media_count_in_post`, `hour_of_day` verwendet.

1. Stärkste positive Korrelation:

follower_count hat eine Korrelation von +0.37 mit likes_log. Das ist die stärkste positive Korrelation in dieser Matrix mit likes_log. Ein Wert von 0.37 deutet auf einen positiven, aber eher moderaten Zusammenhang hin. Das bedeutet, tendenziell haben Profile mit mehr Followern auch Posts mit mehr Likes (bzw. einem höheren Log-Wert der Likes).

2. Stärkste negative Korrelation:

hashtag_count hat eine Korrelation von -0.40 mit likes_log. Dies ist die stärkste negative Korrelation. Ein Wert von -0.40 deutet auf einen moderaten negativen Zusammenhang hin. Das könnte bedeuten, dass Posts mit einer höheren Anzahl von Hashtags tendenziell etwas weniger Likes bekommen.

In [ ]:
# Korrelationen sortieren und Top-3 Features auswählen
corrs = temp_df_corr.corr()['likes_log'].abs().sort_values(ascending=False)
top_features = [col for col in corrs.index if col != 'likes_log'][:3]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, feature in enumerate(top_features):
    sns.regplot(
        x=feature, y='likes_log', data=temp_df_corr,
        scatter_kws={'s': 8, 'alpha': 0.3},
        line_kws={'color': 'red', 'lw': 2},
        ax=axes[i]
    )
    axes[i].set_title(f'Log(Likes) vs. {feature}')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Log(1 + Anzahl Likes)')
    if temp_df_corr[feature].min() > 0 and temp_df_corr[feature].skew() > 2:
        axes[i].set_xscale('log')

plt.tight_layout()
plt.show()

### Beziehung kategorialer Merkmale zu likes_log

- Das Ziel ist es, ein besseres Verständnis dafür zu entwickeln, wie verschiedene kategoriale Merkmale mit der Zielvariablen (likes_log) zusammenhängen. 

Durch die Visualisierung (Boxplots, Linienplots) kann man sehen, ob bestimmte Kategorien eines Merkmals tendenziell höhere oder niedrigere likes_log-Werte aufweisen. Man kann erste Hypothesen über den Einfluss dieser Merkmale auf die Likes bilden.
Es kann auch helfen zu entscheiden, welche Merkmale potenziell wichtig für ein späteres Vorhersagemodell sind. 

In [ ]:
kategoriale_merkmale = [col for col in X.select_dtypes(include=['object', 'bool', 'category']).columns]


if 'hour_of_day' in X.columns and 'hour_of_day' not in kategoriale_merkmale:
    kategoriale_merkmale.append('hour_of_day')
    

for kategorial in kategoriale_merkmale:

    temporaere_plot_df = pd.concat([X[[kategorial]].copy(), y.copy()], axis=1)
    
    plt.figure(figsize=(14, 8))
    
    # Spezialbehandlung für 'account_category'
    if kategorial == 'account_category':
        
        top_n = temporaere_plot_df[kategorial].value_counts().nlargest(10).index # Top 10 Kategorien
        # Alle Kategorien, die nicht zu den Top 10 gehören - werden als 'Andere' zusammengefasst
        temporaere_plot_df[kategorial] = temporaere_plot_df[kategorial].astype(str).apply(lambda x: x if x in top_n else 'Andere')
        order = temporaere_plot_df.groupby(kategorial)['likes_log'].median().sort_values(ascending=False).index
        sns.boxplot(x=kategorial, y='likes_log', data=temporaere_plot_df, order=order, hue=kategorial, palette='viridis', legend=False)
    elif kategorial == 'hour_of_day':
        # Da 'hour_of_day' geordnet ist (0-23), ist ein Linienplot oft aussagekräftiger als einzelne Boxplots für jede Stunde.
        sns.lineplot(x=kategorial, y='likes_log', data=temporaere_plot_df, estimator='median', errorbar='sd')
        plt.xticks(sorted(temporaere_plot_df[kategorial].unique()))
    else:
        # ALLGEMEINER FALL: Für ALLE ANDEREN Merkmale in der Liste "kategoriale_merkmale"

        order = temporaere_plot_df.groupby(kategorial)['likes_log'].median().sort_values(ascending=False).index
        sns.boxplot(x=kategorial, y='likes_log', data=temporaere_plot_df, order=order, hue=kategorial, palette='viridis', legend=False)

    plt.title(f'Log(Likes) nach {kategorial}', fontsize=16)
    plt.xlabel(kategorial, fontsize=14)
    plt.ylabel('Log(1 + Anzahl Likes)', fontsize=14)
    if temporaere_plot_df[kategorial].nunique() > 5 : 
        plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.tight_layout()
    plt.show()


### Interpretation der Ergebnisse:

1. is_verified
    - Verifizierte Accounts haben im Median signifikant höhere Log(Likes)-Werte als nicht verifizierte Accounts. Die Streuung der Likes ist bei verifizierten Accounts ebenfalls größer. Dies legt nahe, dass eine Verifizierung tendenziell mit einer höheren Anzahl an Likes einhergeht.


2. account_category
   - Die Kategorie des Accounts scheint einen deutlichen Einfluss auf die erwarteten Log(Likes) zu haben. Bestimmte Kategorien wie "Athlete" oder potenziell "Entrepreneur" erzielen im Median deutlich höhere Like-Zahlen, während andere wie "Artist" oder "Coach" tendenziell niedrigere Like-Zahlen aufweisen. Die Sortierung nach Median hilft, diese Unterschiede schnell zu erkennen. Die Kategorie "Andere" (falls vorhanden und nicht explizit "Entrepreneur" gemeint ist) zeigt erwartungsgemäß eine hohe Streuung und viele Ausreißer, da sie heterogene Account-Typen zusammenfasst.

3. is_business_account
    - Accounts, die kein Business-Account sind (False), haben im Median tendenziell leicht höhere Log(Likes)-Werte als ausgewiesene Business-Accounts (True). Der Unterschied im Median ist jedoch nicht extrem groß. Beide Gruppen zeigen eine erhebliche Streuung in den Like-Zahlen. Es gibt bei Business-Accounts eine größere Anzahl von Ausreißern im unteren Bereich der Like-Verteilung aufweisen,was bedeuten könnte, dass es zwar einige gut performende Business-Accounts gibt, aber auch eine signifikante Anzahl, die vergleichsweise wenige Likes erhalten.

4. is_professional_account
   - Die Ergebnisse deuten stark darauf hin, dass die Klassifizierung als "Professional Account" ein wichtiger Indikator für höhere Interaktionsraten (Likes) ist. Dies könnte daran liegen, dass professionelle Accounts möglicherweise qualitativ hochwertigeren Content posten, eine größere oder engagiertere Follower-Basis haben oder gezieltere Strategien zur Interaktionssteigerung verfolgen.

5. post_type
    - Der Post-Typ (ob Karussell oder Einzelbild) haben keinen starken, eindeutigen Einfluss auf die durchschnittliche Anzahl der erhaltenen Likes zu haben. Die Performance beider Formate ist im Median und in ihrer Variabilität sehr vergleichbar. 

6. day_of_week
    - Die Wahl des Wochentags für einen Post könnte einen gewissen Einfluss auf die erwarteten Likes haben, wobei Sonntag und Montag die höchsten Median-Log(Likes) aufweisen, was darauf hindeutet, dass Posts an diesen Tagen tendenziell besser performen. Aber Insgesamt sind die Unterschiede im Median zwischen den Wochentagen nicht extrem stark ausgeprägt.

7. month
    - Es scheint saisonale Schwankungen in der Like-Performance zu geben. Die Monate (August, November, Oktober) zeigen tendenziell die höchsten Median-Likes. Monate wie März, Februar weisen eine viele Ausreißer auf, was bedeutet, dass die Like-Zahlen in diesen Monaten stark variieren können.  Die hohe Variabilität in einigen Monaten deutet jedoch darauf hin, dass hier andere Faktoren eine große Rolle spielen.

8. hour_of_day
    - Das Posten um 4 Uhr morgens könnte eine Strategie für potenziell sehr hohe Like-Zahlen sein, ist aber mit einer hohen Unsicherheit verbunden. Die Gründe für diesen Peak könnten vielfältig sein (z.B. internationale Zielgruppen, weniger Konkurrenz im Feed, spezifische Algorithmus-Effekte zu dieser Zeit). Für eine konsistentere, wenn auch vielleicht nicht maximale, Performance könnten andere Tageszeiten geeigneter sein. Die große Streuung um 4 Uhr legt nahe, dass hier weitere Faktoren (z.B. Content-Typ, Account-Bekanntheit) eine sehr große Rolle spielen.

## Hypothese (Gesamtsignifikanz des Modells):

- H0: 
    - Die Gesamtheit der ausgewählten Prädiktoren (is_verified, account_category, follower_count etc.) hat keinen linearen Einfluss auf likes_log.
- H1: 
    - Mindestens einer der ausgewählten Prädiktoren hat einen signifikanten linearen Einfluss auf likes_log. 

### Funktion zur Zusammenfassung der OLS-Ergebnisse definieren


In [ ]:
feature_columns_hypothese = ['follower_count', 'comment_count']

# Account-spezifische Merkmale:
if 'is_verified' in df_gefiltert.columns: feature_columns_hypothese.append('is_verified')
if 'is_professional_account' in df_gefiltert.columns: feature_columns_hypothese.append('is_professional_account')
if 'account_category' in df_gefiltert.columns: feature_columns_hypothese.append('account_category')
if 'is_business_account' in df_gefiltert.columns: feature_columns_hypothese.append('is_business_account') # Auch wenn der Einfluss "leicht negativ" ist, behalten wir es zur Prüfung

# Zeitliche Faktoren:
if 'day_of_week' in df_gefiltert.columns: feature_columns_hypothese.append('day_of_week') 
if 'month' in df_gefiltert.columns: feature_columns_hypothese.append('month')
if 'hour_of_day' in df_gefiltert.columns: feature_columns_hypothese.append('hour_of_day')

# Zusätzliche relevante Metadaten, die oft einen Einfluss haben:
if 'caption_length' in df_gefiltert.columns: feature_columns_hypothese.append('caption_length')
if 'hashtag_count' in df_gefiltert.columns: feature_columns_hypothese.append('hashtag_count')
if 'mention_count' in df_gefiltert.columns: feature_columns_hypothese.append('mention_count')
if 'tagged_user_count' in df_gefiltert.columns: feature_columns_hypothese.append('tagged_user_count')


existing_feature_columns_hypothese = [col for col in feature_columns_hypothese if col in df_gefiltert.columns]
X_hypothese = df_gefiltert[existing_feature_columns_hypothese].copy()
y_hypothese = df_gefiltert['likes_log'].copy()
print("\nAusgewählte Features für X (basierend auf Hypothese):")
X_hypothese.info()


# Identifiziere numerische und kategoriale Spalten in X_hypothese
numerical_cols_hypothese = X_hypothese.select_dtypes(include=np.number).columns.tolist()
categorical_cols_hypothese = X_hypothese.select_dtypes(include=['object', 'bool', 'category']).columns.tolist()


print(f"\nFinale numerische Features für Preprocessing (Hypothese): {numerical_cols_hypothese}")
print(f"Finale kategoriale Features für Preprocessing (Hypothese): {categorical_cols_hypothese}")

# Preprocessing Pipelines
numerical_pipeline_hypo = Pipeline([
    ('scaler', StandardScaler())
])

categorical_pipeline_hypo = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')) 
])

# keine Überschneidungen zwischen numerical_cols und categorical_cols 
common_cols_hypo = list(set(numerical_cols_hypothese) & set(categorical_cols_hypothese))
if common_cols_hypo:
    print(f"WARNUNG: Gemeinsame Spalten in numerical_cols_hypothese und categorical_cols_hypothese gefunden: {common_cols_hypo}")
    for c_col in common_cols_hypo:
        if c_col in numerical_cols_hypothese: numerical_cols_hypothese.remove(c_col)


preprocessor_hypothese = ColumnTransformer([
    ('numerical', numerical_pipeline_hypo, numerical_cols_hypothese),
    ('categorical', categorical_pipeline_hypo, categorical_cols_hypothese)
], remainder='drop')


# Aufteilung in Trainings- und Testsets
X_train_hypo, X_test_hypo, y_train_hypo, y_test_hypo = train_test_split(X_hypothese, y_hypothese, test_size=0.4, random_state=42)

print(f"\nGröße Trainingsdaten X (Hypothese): {X_train_hypo.shape}")
print(f"Größe Testdaten X (Hypothese): {X_test_hypo.shape}")

if X_train_hypo.empty:
    raise ValueError("X_train_hypo ist leer nach dem Splitten. Überprüfe die Daten und Feature-Auswahl.")

In [ ]:
def summarize_ols_regression_results(results, X_test_data, y_test_data, feature_names=None):
    """
    Erstellt eine Zusammenfassung der OLS-Regressionsergebnisse,
    berechnet Testmetriken und visualisiert Ergebnisse.

    Args:
        results (statsmodels.regression.linear_model.RegressionResultsWrapper): 
            Das gefittete OLS-Modell von statsmodels.
        X_test_data (pd.DataFrame oder np.array): 
            Die Testdaten für die Features (bereits prozessiert und mit Konstante).
        y_test_data (pd.Series oder np.array): 
            Die wahren Werte der Zielvariable für die Testdaten.
        feature_names (list, optional): 
            Liste der Feature-Namen für eine bessere Darstellung in der Summary,
    """
    if results is None:
        print("Modell konnte nicht trainiert werden. Keine Ergebnisse zum Anzeigen.")
        return

    print("OLS Regression Results Summary:")
    print("==============================")
    print(results.summary(xname=feature_names if feature_names else list(X_test_data.columns)))
    print("\nParameterschätzungen (Koeffizienten):")
    print("------------------------------------")
    print(results.params)

    # Vorhersagen auf Testdaten
    y_pred_test = results.predict(X_test_data)

    # Metriken für Testdaten (auf der Skala von log-Likes)
    mse_test = mean_squared_error(y_test_data, y_pred_test)
    mae_test = mean_absolute_error(y_test_data, y_pred_test)
    r2_test_sm = r2_score(y_test_data, y_pred_test)

    print("\n--- Metriken auf Testdaten (log-transformierte Likes) ---")
    print(f"Mean Squared Error (MSE): {mse_test:.4f}")
    print(f"Mean Absolute Error (MAE): {mae_test:.4f}")
    print(f"R-squared (R²): {r2_test_sm:.4f} (berechnet für Testdaten)")
    print(f"AIC: {results.aic:.2f}")
    print(f"BIC: {results.bic:.2f}")

    # Rücktransformation für interpretierbare Metriken
    target_variable_name = y_test_data.name if hasattr(y_test_data, 'name') else 'likes_log' # Standardname falls unbekannt

    if target_variable_name == 'likes_log':
        y_test_orig = np.expm1(y_test_data)
        y_pred_test_orig = np.expm1(y_pred_test)
        y_pred_test_orig[y_pred_test_orig < 0] = 0 

        print("\n--- Metriken auf Testdaten (Original-Skala der Likes) ---")
        print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test_orig, y_pred_test_orig):.2f} Likes")
        print(f"Root Mean Squared Error (RMSE): {np.sqrt(mean_squared_error(y_test_orig, y_pred_test_orig)):.2f} Likes")
        print(f"R-squared (R²) auf Originalskala: {r2_score(y_test_orig, y_pred_test_orig):.4f}")

    # Visualisierung der Vorhersagen vs. tatsächliche Werte
    plt.figure(figsize=(10, 6))
    plt.scatter(y_test_data, y_pred_test, alpha=0.5, label='Vorhersagen')
    plt.plot([y_test_data.min(), y_test_data.max()], [y_test_data.min(), y_test_data.max()], '--', color='red', lw=2, label='Perfekte Vorhersage')
    plt.xlabel(f"Tatsächliche {target_variable_name}")
    plt.ylabel(f"Vorhergesagte {target_variable_name}")
    plt.title(f"Tatsächliche vs. Vorhergesagte {target_variable_name} (Testdaten - OLS)")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Residuenplot
    residuals = y_test_data.reset_index(drop=True) - pd.Series(y_pred_test).reset_index(drop=True) # Stelle Index-Konsistenz sicher
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=y_pred_test, y=residuals, alpha=0.5)
    plt.axhline(0, color='red', linestyle='--')
    plt.xlabel("Vorhergesagte Werte")
    plt.ylabel("Residuen (Tatsächlich - Vorhergesagt)")
    plt.title("Residuenplot")
    plt.grid(True)
    plt.show()

### Training eines auf der Hypothese basierenden Modells (Lineare Regression mit statsmodels) 

In [ ]:
print("Starte Daten-Preprocessing für das hypothesenbasierte OLS-Modell...")
try:
    X_train_processed_hypo_np = preprocessor_hypothese.fit_transform(X_train_hypo)
    X_test_processed_hypo_np = preprocessor_hypothese.transform(X_test_hypo)
    print("Daten-Preprocessing abgeschlossen.")

    transformed_feature_names_hypo = []
    
    # Numerische Features
    num_transformer_hypo = preprocessor_hypothese.named_transformers_.get('numerical')
    if num_transformer_hypo and numerical_cols_hypothese: # numerical_cols_hypothese ist die Liste der numerischen Spalten vor OHE
        # Die Namen bleiben nach Skalierung/Imputation gleich
        transformed_feature_names_hypo.extend(numerical_cols_hypothese)

    # One-Hot-kodierte Features
    cat_transformer_hypo = preprocessor_hypothese.named_transformers_.get('categorical')
    if cat_transformer_hypo and categorical_cols_hypothese: # categorical_cols_hypothese ist die Liste der kategorialen Spalten vor OHE
        onehot_step_hypo = cat_transformer_hypo.named_steps.get('onehot')
        if onehot_step_hypo and hasattr(onehot_step_hypo, 'get_feature_names_out'):
            try:
                # Übergebe die ursprünglichen kategorialen Spaltennamen an get_feature_names_out
                onehot_feature_names_hypo = onehot_step_hypo.get_feature_names_out(categorical_cols_hypothese)
                transformed_feature_names_hypo.extend(list(onehot_feature_names_hypo))
            except Exception as e_gn:
                print(f"Fehler beim Holen der OHE Feature Namen mit get_feature_names_out: {e_gn}")
                pass 
        else:
            if categorical_cols_hypothese:
                 print("Warnung: Konnte Feature-Namen von OneHotEncoder nicht automatisch beziehen (get_feature_names_out nicht verfügbar/erfolgreich).")

    # DataFrame erstellen
    # Wenn die Feature-Namen nicht korrekt extrahiert werden konnten, ist es besser,
    # die Anzahl der Spalten direkt vom NumPy-Array zu nehmen und generische Namen zu verwenden.
    if X_train_processed_hypo_np.shape[1] == len(transformed_feature_names_hypo) and transformed_feature_names_hypo:
        X_train_processed_hypo_df = pd.DataFrame(X_train_processed_hypo_np, columns=transformed_feature_names_hypo, index=X_train_hypo.index)
        X_test_processed_hypo_df = pd.DataFrame(X_test_processed_hypo_np, columns=transformed_feature_names_hypo, index=X_test_hypo.index)
        print(f"Verwende {len(transformed_feature_names_hypo)} extrahierte Feature-Namen.")
    else:
        num_cols_after_transform = X_train_processed_hypo_np.shape[1]
        print(f"Warnung: Anzahl der extrahierten Feature-Namen ({len(transformed_feature_names_hypo)}) stimmt nicht mit der Spaltenanzahl der prozessierten Daten ({num_cols_after_transform}) überein oder Namen sind leer. Verwende generische Namen.")
        fallback_cols = [f"feature_{i}" for i in range(num_cols_after_transform)]
        X_train_processed_hypo_df = pd.DataFrame(X_train_processed_hypo_np, columns=fallback_cols, index=X_train_hypo.index)
        X_test_processed_hypo_df = pd.DataFrame(X_test_processed_hypo_np, columns=fallback_cols, index=X_test_hypo.index)
        transformed_feature_names_hypo = fallback_cols # Wichtig für die Summary später


    # eine Konstante für den Intercept 
    X_train_ols_hypo = sm.add_constant(X_train_processed_hypo_df.reset_index(drop=True), has_constant='add')
    X_test_ols_hypo = sm.add_constant(X_test_processed_hypo_df.reset_index(drop=True), has_constant='add')

    print("\nStarte OLS-Modelltraining (hypothesenbasiert)...")
    ols_model_hypo = sm.OLS(y_train_hypo.reset_index(drop=True), X_train_ols_hypo)
    ols_results_hypo = ols_model_hypo.fit()
    print("OLS-Modelltraining (hypothesenbasiert) abgeschlossen.")

except Exception as e:
    print(f"Fehler während des Preprocessings oder OLS-Modelltrainings (hypothesenbasiert): {e}")


### Evaluierung des hypothesenbasierten Modells (OLS-Regression) 

In [ ]:
print("\nStarte OLS-Modellevaluierung (hypothesenbasiert)...")
if 'ols_results_hypo' in locals() and ols_results_hypo is not None and \
   'X_test_ols_hypo' in locals() and X_test_ols_hypo is not None and \
   'y_test_hypo' in locals() and 'transformed_feature_names_hypo' in locals():
    summarize_ols_regression_results(ols_results_hypo, X_test_ols_hypo, y_test_hypo, feature_names=list(X_test_ols_hypo.columns))
else:
    print("Keine OLS-Ergebnisse (hypothesenbasiert), Testdaten oder Feature-Namen zum Evaluieren vorhanden. Überprüfe den Trainingsschritt.")

#### Interpretation der Ergebnisse:

- Der Prob (F-statistic)-Wert von 7.12e-241 ist extrem klein (praktisch Null).
Schlussfolgerung: Wir verwerfen die Nullhypothese (H0) für die Gesamtsignifikanz. Das Modell als Ganzes ist statistisch hoch signifikant. Die ausgewählten Features leisten gemeinsam einen signifikanten Beitrag zur Erklärung der Varianz in likes_log.

- R-squared: 0.680: Etwa 68.0% der Varianz in der log-transformierten Like-Anzahl (likes_log) können durch die im Modell enthaltenen Features erklärt werden.

- Adj. R-squared: 0.666: Das adjustierte R-Quadrat, das die Anzahl der Prädiktoren im Modell berücksichtigt, ist mit 0.666 nur geringfügig niedriger. Dies deutet darauf hin, dass die meisten der 51 Prädiktoren (Df Model) tatsächlich zur Erklärungskraft beitragen. Für Social-Media-Daten ist dies ein guter Wert, da die Like-Zahlen oft von vielen weiteren, nicht erfassten Faktoren beeinflusst werden.


#### Bestätigte Haupteinflüsse: 

- Die professionelle Ausrichtung (is_professional_account_True) und Verifizierung (is_verified_True) eines Accounts sowie die Follower-Anzahl (follower_count) sind die dominantesten positiven Prädiktoren für höhere Like-Zahlen. Die account_category spielt eine differenzierte und signifikante Rolle, wobei einige Kategorien stark positiv (z.B. Athlete, Brand) und andere stark negativ (z.B. Musician, Entrepreneur im Modellkontext) abschneiden.